In [3]:
import pandas as pd
import numpy as np

shipment=pd.read_csv('shpiment_cleaned2.csv')
shipment_leg=pd.read_csv('shipment_leg_cleaned2.csv')
ports=pd.read_csv('ports_cleaned.csv')
carriers=pd.read_csv('carriers_cleaned.csv')
carbon_log=pd.read_csv('carbon_logs.csv')
telemetry=pd.read_csv('telemetry.csv')
tariffs_customs=pd.read_csv('tariffs_customs.csv')

In [4]:
shipment_leg = shipment_leg.merge(
    ports[[
        "PortID",
        "PortName",
        "Country",
        "Region",
        "MasterPortID",
        "PortClusterID",
        "HandlingFee_USD",
        "Capacity_TEU"
    ]].rename(columns={
        "PortID": "OriginPortID",
        "PortName": "OriginPortName",
        "Country": "OriginPortCountry",
        "Region": "OriginPortRegion",
        "MasterPortID": "OriginMasterPortID",
        "PortClusterID": "OriginPortClusterID",
        "HandlingFee_USD": "OriginHandlingFee_USD",
        "Capacity_TEU": "OriginCapacity_TEU"
    }),
    on="OriginPortID",
    how="left"
)

In [5]:
shipment_leg = shipment_leg.merge(
    ports[[
        "PortID",
        "PortName",
        "Country",
        "Region",
        "MasterPortID",
        "PortClusterID",
        "HandlingFee_USD",
        "Capacity_TEU"
    ]].rename(columns={
        "PortID": "DestinationPortID",
        "PortName": "DestinationPortName",
        "Country": "DestinationPortCountry",
        "Region": "DestinationPortRegion",
        "MasterPortID": "DestinationMasterPortID",
        "PortClusterID": "DestinationPortClusterID",
        "HandlingFee_USD": "DestinationHandlingFee_USD",
        "Capacity_TEU": "DestinationCapacity_TEU"
    }),
    on="DestinationPortID",
    how="left"
)

In [6]:
shipment_leg["InvalidOriginPortFlag"] = shipment_leg["OriginPortName"].isna()
shipment_leg["InvalidDestinationPortFlag"] = shipment_leg["DestinationPortName"].isna()

In [7]:
shipment_leg = shipment_leg.merge(
    carriers[[
        "CarrierID",
        "CarrierName",
        "Mode",
        "SLA_Days",
        "LatePenalty_USD",
        "CO2_Factor",
        "ReportedESGRating",
        "ESGScore",
        "ESGTier",
        "CO2FactorCategory"
    ]].rename(columns={
        "Mode": "CarrierMode"
    }),
    on="CarrierID",
    how="left"
)

In [8]:
shipment_leg["InvalidCarrierFlag"] = shipment_leg["CarrierName"].isna()

In [9]:
shipment_leg["CarrierModeMismatchFlag"] = (
    shipment_leg["CarrierMode"].notna() &
    (shipment_leg["Mode"] != shipment_leg["CarrierMode"])
)

In [10]:
print(shipment_leg["CarrierModeMismatchFlag"].sum())

9675


In [11]:
shipment_leg["LegExceedsCarrierSLAFlag"] = (
    shipment_leg["LegDurationDays"].notna() &
    shipment_leg["SLA_Days"].notna() &
    (shipment_leg["LegDurationDays"] > shipment_leg["SLA_Days"])
)

In [12]:
print(shipment_leg["LegExceedsCarrierSLAFlag"].sum())

6688


In [13]:
shipment_leg = shipment_leg.sort_values(["ShipmentID", "LegSequence"]).reset_index(drop=True)

In [14]:
shipment_leg["PreviousDestinationPortClusterID"] = (
    shipment_leg.groupby("ShipmentID")["DestinationPortClusterID"]
    .shift(1)
)

In [15]:
shipment_leg["RouteBreakClusterFlag"] = (
    (shipment_leg["LegSequence"] > 1) &
    (shipment_leg["OriginPortClusterID"] != shipment_leg["PreviousDestinationPortClusterID"])
)

In [16]:
used_ports = set(shipment_leg["OriginPortID"]).union(set(shipment_leg["DestinationPortID"]))

ports["UsedInLegsFlag"] = ports["PortID"].isin(used_ports)
ports["UnusedPortFlag"] = ~ports["UsedInLegsFlag"]

In [17]:
print(ports[ports["UnusedPortFlag"] == True])

   PortID            PortName         Country  HandlingFee_USD  Capacity_TEU  \
1     P02    Port of Shenzhen           China              800      30000000   
8     P09   Port of Singapore       Singapore              750      37000000   
10    P11     Port of Hamburg         Germany             1200       9000000   
13    P14  Port of Felixstowe  United Kingdom             1050       4000000   
14    P15     Port of Antwerp         Belgium             1080      12000000   

    Latitude  Longitude  NegativeHandlingFeeFlag  NegativeCapacityFlag  \
1    22.5431   114.0579                    False                 False   
8     1.3521   103.8198                    False                 False   
10   53.5511     9.9937                    False                 False   
13   51.9632     1.3512                    False                 False   
14   51.2194     4.4025                    False                 False   

    MissingHandlingFeeFlag  ...  Region                        PortFullNam

In [18]:
used_carriers = set(shipment_leg["CarrierID"])

carriers["UsedInLegsFlag"] = carriers["CarrierID"].isin(used_carriers)
carriers["UnusedCarrierFlag"] = ~carriers["UsedInLegsFlag"]

In [19]:
print(carriers[carriers["UnusedCarrierFlag"] == True])

Empty DataFrame
Columns: [Unnamed: 0, CarrierID, CarrierName, Mode, SLA_Days, LatePenalty_USD, CO2_Factor, ReportedESGRating, NegativeSLADaysFlag, NegativePenaltyFlag, NegativeCO2FactorFlag, MissingSLAFlag, MissingPenaltyFlag, MissingCO2FactorFlag, ESGScore, ESGTier, CO2FactorCategory, PenaltyCategory, SLACategory, UsedInLegsFlag, UnusedCarrierFlag]
Index: []

[0 rows x 21 columns]


In [20]:
shipment_leg.to_csv('shipment_leg_cleaned2.csv', index=False)
ports.to_csv('ports_cleaned.csv', index=False)
carriers.to_csv('carriers_cleaned.csv', index=False)